# 🤖 Machine Learning Pose Classification Tutorial

Tutorial lengkap untuk training dan testing model klasifikasi pose.

## Setup

In [1]:
# Install dependencies (uncomment jika belum install)
# !pip install opencv-python mediapipe scikit-learn pandas matplotlib seaborn

In [2]:
# Import libraries
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pickle

print("✅ All libraries imported successfully!")

ModuleNotFoundError: No module named 'cv2'

## 1️⃣ Feature Extraction

Ekstraksi pose landmarks dari gambar menggunakan MediaPipe.

In [ ]:
class PoseFeatureExtractor:
    def __init__(self):
        self.mp_pose = mp.solutions.pose
        self.pose = self.mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)
    
    def extract_landmarks(self, image_path):
        """Extract pose landmarks from image"""
        image = cv2.imread(image_path)
        if image is None:
            return None
        
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = self.pose.process(image_rgb)
        
        if results.pose_landmarks:
            landmarks = []
            for landmark in results.pose_landmarks.landmark:
                landmarks.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
            return np.array(landmarks)
        
        return None
    
    def visualize_pose(self, image_path):
        """Visualize detected pose landmarks"""
        image = cv2.imread(image_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = self.pose.process(image_rgb)
        
        if results.pose_landmarks:
            mp_drawing = mp.solutions.drawing_utils
            annotated_image = image.copy()
            mp_drawing.draw_landmarks(
                annotated_image,
                results.pose_landmarks,
                self.mp_pose.POSE_CONNECTIONS
            )
            
            # Display
            plt.figure(figsize=(12, 6))
            plt.subplot(1, 2, 1)
            plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            plt.title('Original Image')
            plt.axis('off')
            
            plt.subplot(1, 2, 2)
            plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
            plt.title('Pose Landmarks')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()

# Create extractor
extractor = PoseFeatureExtractor()
print("✅ Feature extractor created!")

In [ ]:
# Test dengan sample image (ganti dengan path gambar Anda)
sample_image = 'dataset/thinking/image_001.jpg'  # Ubah path sesuai kebutuhan

# Visualize
extractor.visualize_pose(sample_image)

# Extract features
features = extractor.extract_landmarks(sample_image)
print(f"\n📊 Extracted {len(features)} features from image")
print(f"Feature shape: {features.shape}")

## 2️⃣ Load Dataset

Load semua gambar training dari direktori.

In [ ]:
import os

def load_dataset(data_dir='dataset'):
    """Load dataset from directory"""
    features_list = []
    labels_list = []
    
    for label in os.listdir(data_dir):
        label_dir = os.path.join(data_dir, label)
        
        if not os.path.isdir(label_dir):
            continue
        
        print(f"Loading {label}...")
        count = 0
        
        for image_file in os.listdir(label_dir):
            if not image_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            
            image_path = os.path.join(label_dir, image_file)
            features = extractor.extract_landmarks(image_path)
            
            if features is not None:
                features_list.append(features)
                labels_list.append(label)
                count += 1
        
        print(f"  ✓ Loaded {count} images")
    
    return np.array(features_list), np.array(labels_list)

# Load dataset
print("📂 Loading dataset...\n")
X, y = load_dataset('dataset')  # Ubah 'dataset' ke path Anda

print(f"\n✅ Dataset loaded!")
print(f"   Total samples: {len(X)}")
print(f"   Feature dimension: {X.shape[1]}")
print(f"   Classes: {np.unique(y)}")

In [ ]:
# Dataset statistics
df = pd.DataFrame({'Label': y})
class_counts = df['Label'].value_counts()

print("\n📊 Class Distribution:")
print(class_counts)

# Visualize
plt.figure(figsize=(10, 6))
class_counts.plot(kind='bar', color='skyblue')
plt.title('Dataset Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Pose Class')
plt.ylabel('Number of Samples')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3️⃣ Prepare Data

Split data menjadi training dan testing sets.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42,
    stratify=y_encoded
)

print("✅ Data prepared!")
print(f"   Training samples: {len(X_train)}")
print(f"   Testing samples: {len(X_test)}")
print(f"   Train/Test ratio: {len(X_train)/len(X_test):.2f}")

## 4️⃣ Train Models

Train multiple ML models dan compare performance.

In [ ]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Define models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

# Train and evaluate
results = {}

print("🎯 Training models...\n")

for name, model in models.items():
    print(f"Training {name}...")
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'predictions': y_pred
    }
    
    print(f"  ✓ Accuracy: {accuracy*100:.2f}%\n")

print("✅ All models trained!")

In [ ]:
# Model comparison
model_names = list(results.keys())
accuracies = [results[m]['accuracy']*100 for m in model_names]

plt.figure(figsize=(10, 6))
bars = plt.bar(model_names, accuracies, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.ylim(0, 105)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 5️⃣ Detailed Evaluation

Analisis performa model terbaik.

In [ ]:
# Select best model
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
best_model = results[best_model_name]['model']
y_pred = results[best_model_name]['predictions']

print(f"🏆 Best Model: {best_model_name}")
print(f"   Accuracy: {results[best_model_name]['accuracy']*100:.2f}%\n")

# Classification report
print("📋 Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

## 6️⃣ Save Model

Save trained model untuk digunakan nanti.

In [ ]:
# Save model
model_data = {
    'model': best_model,
    'label_encoder': label_encoder,
    'model_name': best_model_name,
    'accuracy': results[best_model_name]['accuracy']
}

with open('pose_classifier.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("💾 Model saved to 'pose_classifier.pkl'")
print(f"   Model: {best_model_name}")
print(f"   Accuracy: {results[best_model_name]['accuracy']*100:.2f}%")

## 7️⃣ Test Prediction

Test model pada gambar baru.

In [ ]:
def predict_image(image_path, model, label_encoder, extractor):
    """Predict pose from image"""
    # Extract features
    features = extractor.extract_landmarks(image_path)
    
    if features is None:
        return None, 0.0
    
    # Predict
    features = features.reshape(1, -1)
    prediction = model.predict(features)[0]
    
    # Get confidence
    if hasattr(model, 'predict_proba'):
        probabilities = model.predict_proba(features)[0]
        confidence = np.max(probabilities) * 100
    else:
        confidence = 100.0
    
    # Decode label
    pose_name = label_encoder.inverse_transform([prediction])[0]
    
    return pose_name, confidence

# Test prediction
test_image = 'dataset/thinking/image_001.jpg'  # Ubah path

pose, conf = predict_image(test_image, best_model, label_encoder, extractor)

if pose:
    print(f"🎯 Prediction: {pose.upper()}")
    print(f"📊 Confidence: {conf:.1f}%")
    
    # Visualize
    extractor.visualize_pose(test_image)
else:
    print("❌ No pose detected")

## 🎉 Congratulations!

Anda telah berhasil:
- ✅ Ekstraksi features dari gambar
- ✅ Load dan prepare dataset
- ✅ Train multiple ML models
- ✅ Evaluate dan compare models
- ✅ Save trained model
- ✅ Make predictions

### Next Steps:

1. Collect lebih banyak data training
2. Experiment dengan hyperparameters
3. Add feature engineering
4. Try deep learning models
5. Deploy ke production!

### Resources:

- [MediaPipe Documentation](https://google.github.io/mediapipe/)
- [scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html)
- [OpenCV Tutorials](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)